In [7]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [8]:
# 1. 讀取整理好的資料
file_path = 'C:\\Users\\170134\\Desktop\\新增資料夾\\cleaned_data_for_analysis.csv'
print(f"正在讀取資料: {file_path} ...")
df = pd.read_csv(file_path)

# --- 資料前處理 (重要！) ---
# 1. 確保 ESG 分數是數字 (有些資料庫可能是文字)
# 如果您的資料是 'A+', 'B' 等級，需要先進行 Mapping 轉換，這裡假設已經是分數
numeric_cols = ['Return', 'Total_ESG', 'E_Score', 'S_Score', 'G_Score']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

正在讀取資料: C:\Users\170134\Desktop\新增資料夾\cleaned_data_for_analysis.csv ...


In [9]:

print(f"原始資料樣本數: {len(df)} 筆")
# 2. 去除極端值 (Winsorization) - 這是財務研究的標準動作
# 因為有些公司年報酬率可能飆到 500% 或跌到 -90%，這些異常值會扭曲迴歸線
# 我們只保留 1% ~ 99% 中間的數據
lower = df['Return'].quantile(0.01)
upper = df['Return'].quantile(0.99)
df = df[(df['Return'] >= lower) & (df['Return'] <= upper)]


print(f"有效分析樣本數: {len(df)} 筆")

原始資料樣本數: 8688 筆
有效分析樣本數: 8514 筆


In [13]:
# 2. 確保資料有按照順序排列 (這一步超重要！)
# 先排公司，再排年份。這樣 shift 才會是 "把去年的資料搬到今年"
df = df.sort_values(by=['Company', 'Year'], ascending=[True, True])

# 3. 製作 Lag 變數 (遞延 1 年)
# 語法解釋: 對每一家公司(groupby)，把 ESG 分數往下移一格(shift 1)
lag_cols = ['Total_ESG', 'E_Score', 'S_Score', 'G_Score']

print("正在建立遞延變數 (Lag 1)...")
for col in lag_cols:
    # 新欄位名稱加上 _Lag1
    df[f'{col}_Lag1'] = df.groupby('Company')[col].shift(1)

# --- 檢查一下有沒有做對 ---
# 我們挑一家公司出來看 (例如第一家)
print("\n--- 資料檢查 (Before & After) ---")
example_company = df['Company'].iloc[0]
print(df[df['Company'] == example_company][['Year', 'Total_ESG', 'Total_ESG_Lag1']].head())

# 妳會發現：
# 2014 年的 Total_ESG_Lag1 變成了 NaN (因為沒有 2013 的資料)
# 2015 年的 Total_ESG_Lag1 就是 2014 年原本的分數 -> 成功！

# 4. 去除空值
# 因為做了 shift(1)，每家公司的第一年都會變成空值，必須刪掉才能跑迴歸
df_lag = df.dropna()
print(f"\n去除空值後樣本數: {len(df_lag)} 筆")

正在建立遞延變數 (Lag 1)...

--- 資料檢查 (Before & After) ---
   Year  Total_ESG  Total_ESG_Lag1
0  2022      61.79             NaN
1  2023      67.87           61.79
2  2024      61.98           67.87
3  2025      61.86           61.98

去除空值後樣本數: 6223 筆


In [15]:
# ========================================================
# 第一階段：ESG 總分分析 (Total_ESG)
# ========================================================

print("\n" + "="*50)
print("  分析 1: ESG 總分對投資報酬率的影響")
print("="*50)

# 模型 A: 簡單迴歸 (Naive Model)
# 假設：只有 ESG 影響報酬率
# 公式: Return = Intercept + Total_ESG
model_a = smf.ols('Return ~ Total_ESG_Lag1', data=df).fit()

# 模型 B: 加入控制變數 (Controlled Model)
# 假設：除了 ESG，還有「產業」和「年度」的影響
# C(Industry) 代表把產業當作類別變數 (Dummy Variable)
# C(Year) 代表把年度當作類別變數
# 公式: Return = Intercept + Total_ESG + 產業效應 + 年度效應
model_b = smf.ols('Return ~ Total_ESG_Lag1 + C(Industry) + C(Year)', data=df).fit()

# 印出比較結果 (我們只看重點數據)
print("\n--- 模型 A (未控制) vs 模型 B (有控制) ---")
print(f"模型 A (未控制) ESG 係數: {model_a.params['Total_ESG_Lag1']:.4f} (P值: {model_a.pvalues['Total_ESG_Lag1']:.4f})")
print(f"模型 B (有控制) ESG 係數: {model_b.params['Total_ESG_Lag1']:.4f} (P值: {model_b.pvalues['Total_ESG_Lag1']:.4f})")

# print(model_a.summary())
# print(model_b.summary())

if model_b.pvalues['Total_ESG_Lag1'] < 0.05:
    print(">> 結論: 在排除產業與年度影響後，ESG 對股價仍有顯著影響！")
else:
    print(">> 結論: 在排除產業與年度影響後，ESG 對股價沒有顯著影響 (可能是產業紅利造成的假象)。")


  分析 1: ESG 總分對投資報酬率的影響

--- 模型 A (未控制) vs 模型 B (有控制) ---
模型 A (未控制) ESG 係數: 0.0839 (P值: 0.1400)
模型 B (有控制) ESG 係數: 0.1052 (P值: 0.0562)
>> 結論: 在排除產業與年度影響後，ESG 對股價沒有顯著影響 (可能是產業紅利造成的假象)。


In [ ]:
# ========================================================
# 第二階段：ESG 個別構面分析 (E, S, G 分開看)
# ========================================================

print("\n" + "="*50)
print("  分析 2: E、S、G 個別分數的影響力 PK")
print("="*50)

# 模型 C: 拆解 ESG (Disaggregated Model)
# 公式: Return = Intercept + E + S + G + 產業 + 年度
model_c = smf.ols('Return ~ E_Score_Lag1 + S_Score_Lag1 + G_Score_Lag1 + C(Industry) + C(Year)', data=df).fit()

# 顯示完整報表 (可以看到 E, S, G 誰的 P值最小)
# print(model_c.summary())

# 自動判讀誰最強
params = model_c.params[['E_Score_Lag1', 'S_Score_Lag1', 'G_Score_Lag1']]
pvals = model_c.pvalues[['E_Score_Lag1', 'S_Score_Lag1', 'G_Score_Lag1']]

print("\n--- 個別構面影響力解析 ---")
for component in ['E_Score_Lag1', 'S_Score_Lag1', 'G_Score_Lag1']:
    coef = params[component]
    p_val = pvals[component]
    significance = "顯著 (***)" if p_val < 0.01 else "顯著 (*)" if p_val < 0.05 else "不顯著"
    print(f"{component}: 係數 {coef:.4f} | P值 {p_val:.4f} -> {significance}")

best_component = params.idxmax()
print(f"\n>> 初步觀察: 對報酬率正向影響最大的是 [{best_component}]")


  分析 2: E、S、G 個別分數的影響力 PK
                            OLS Regression Results                            
Dep. Variable:                 Return   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     21.04
Date:                  週六, 07 二月 2026   Prob (F-statistic):          6.87e-134
Time:                        00:00:15   Log-Likelihood:                -31895.
No. Observations:                6223   AIC:                         6.387e+04
Df Residuals:                    6184   BIC:                         6.413e+04
Df Model:                          38                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------